# Lección 3: IMPLEMENTACIÓN DE RN EN PYTHON
## A. Modelo funcional para clasificar imágenes (Fashion-MNIST)

El dataset **Fashion-MNIST** es un estándar en la industria para practicar. Contiene 70,000 imágenes en escala de grises de artículos de ropa (zapatillas, camisetas, bolsos, etc.), clasificadas en 10 categorías distintas. Cada imagen tiene una resolución de 28x28 píxeles.

Para este problema, construiremos una Red Neuronal Convolucional (CNN) sencilla. Un paso crucial antes de entrenar es la **normalización**: los valores de los píxeles van de 0 a 255, así que los dividiremos entre 255 para que queden en un rango de 0 a 1. Esto le facilita la vida al optimizador y hace que la red aprenda más rápido.

In [ ]:
# ── CELDA DE CARGA DE DATOS (ejecutar siempre primero) ───────────────────────
# Esta celda carga y prepara el dataset. Todas las celdas de esta lección
# dependen de las variables aquí definidas.

import tensorflow as tf
from tensorflow.keras import datasets, layers, models
import matplotlib.pyplot as plt

# 1. Cargar el dataset Fashion-MNIST
(train_images, train_labels), (test_images, test_labels) = datasets.fashion_mnist.load_data()

# 2. Normalizar los datos (valores entre 0 y 1)
train_images = train_images / 255.0
test_images  = test_images  / 255.0

print(f"✅ Datos cargados correctamente.")
print(f"   Entrenamiento: {train_images.shape} | Etiquetas: {train_labels.shape}")
print(f"   Prueba:        {test_images.shape}  | Etiquetas: {test_labels.shape}")

In [ ]:
# ── A. Entrenamiento del modelo base ─────────────────────────────────────────

# 3. Definir la arquitectura del modelo (CNN)
# Usamos Input + Reshape para consistencia con el resto del proyecto
model_base = models.Sequential([
    layers.Input(shape=(28, 28)),
    layers.Reshape((28, 28, 1)),  # Adaptamos (28,28) -> (28,28,1) para Conv2D

    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='softmax')  # 10 categorías de ropa
])

# 4. Compilar el modelo
model_base.compile(optimizer='adam',
                   loss='sparse_categorical_crossentropy',
                   metrics=['accuracy'])

# 5. Entrenar el modelo
print("Iniciando entrenamiento... (esto puede tardar unos minutos)")
history = model_base.fit(
    train_images, train_labels,
    epochs=5,
    batch_size=128,          # Batch más grande = épocas más rápidas
    validation_split=0.2,
    verbose=1
)

# 6. Evaluar con los datos de prueba
test_loss, test_acc = model_base.evaluate(test_images, test_labels, verbose=0)
print(f"\nPrecisión final en datos nuevos (Test): {test_acc:.4f}")

### Análisis de los resultados

* **Importancia de la normalización:** Escalar los datos es innegociable en el procesamiento de imágenes. Si le pasamos los píxeles crudos (hasta 255), los pesos de la red oscilarían demasiado y el modelo tardaría una eternidad en aprender algo útil.
* **Elección de la pérdida:** Utilizamos `sparse_categorical_crossentropy` en lugar de `categorical_crossentropy` porque nuestras etiquetas vienen como números enteros (ej. 9 para botines, 0 para camisetas) en vez de vectores binarios.
* **Tiempos y Rendimiento:** Con solo 5 épocas, el modelo ya debería rondar el 85-90% de precisión. Si vemos que la precisión de entrenamiento (`accuracy`) sube, pero la de validación (`val_accuracy`) se estanca o baja, significa que el modelo está empezando a memorizar (sobreajuste).

## B. Entrenar y validar el modelo usando diferentes hiperparámetros

Afinar un modelo es muy parecido a afinar un instrumento musical. Los **hiperparámetros** son todas esas configuraciones que nosotros definimos *antes* de empezar el entrenamiento (a diferencia de los pesos, que el modelo aprende por sí solo). 

Algunos de los más comunes y que más impacto tienen son:
* **Tasa de aprendizaje (Learning Rate):** Define qué tan grandes son los pasos que da el modelo para corregir sus errores. Si es muy grande, no aprende bien; si es muy pequeña, tarda una eternidad.
* **Tamaño de lote (Batch Size):** Cuántas imágenes agrupa y procesa a la vez antes de actualizar sus cálculos.
* **Optimizador:** El algoritmo matemático que guía el aprendizaje.

Para ver esto en la práctica, vamos a entrenar dos modelos idénticos en su arquitectura, pero con configuraciones de entrenamiento diferentes, y luego graficaremos cuál aprende mejor.

In [ ]:
# ── B. Comparación de hiperparámetros ────────────────────────────────────────
# NOTA: Requiere que la celda de carga de datos haya sido ejecutada primero.

from tensorflow.keras import optimizers
import matplotlib.pyplot as plt

# Función para no repetir el código de la arquitectura
def crear_modelo():
    return models.Sequential([
        layers.Input(shape=(28, 28)),
        layers.Reshape((28, 28, 1)),
        layers.Conv2D(32, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Flatten(),
        layers.Dense(64, activation='relu'),
        layers.Dense(10, activation='softmax')
    ])

# ---------------------------------------------------------
# MODELO 1: Configuración "Estándar" (Optimizador Adam, Lote de 32)
# ---------------------------------------------------------
print("Entrenando Modelo 1 (Adam, Batch=32)...")
modelo_1 = crear_modelo()
modelo_1.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
hist_1 = modelo_1.fit(train_images, train_labels, epochs=5, batch_size=32,
                      validation_split=0.2, verbose=0)

# ---------------------------------------------------------
# MODELO 2: Configuración "Alternativa" (Optimizador SGD clásico, Lote de 128)
# ---------------------------------------------------------
print("Entrenando Modelo 2 (SGD, Batch=128)...")
modelo_2 = crear_modelo()
opt_sgd = optimizers.SGD(learning_rate=0.01, momentum=0.9)
modelo_2.compile(optimizer=opt_sgd, loss='sparse_categorical_crossentropy', metrics=['accuracy'])
hist_2 = modelo_2.fit(train_images, train_labels, epochs=5, batch_size=128,
                      validation_split=0.2, verbose=0)

# ---------------------------------------------------------
# EVALUACIÓN Y GRÁFICOS
# ---------------------------------------------------------
_, acc_1 = modelo_1.evaluate(test_images, test_labels, verbose=0)
_, acc_2 = modelo_2.evaluate(test_images, test_labels, verbose=0)

print(f"\nPrecisión final Modelo 1 (Adam)  en Test: {acc_1:.4f}")
print(f"Precisión final Modelo 2 (SGD)   en Test: {acc_2:.4f}")

plt.figure(figsize=(8, 5))
plt.plot(hist_1.history['val_accuracy'], label='Modelo 1 (Adam / Batch 32)', marker='o')
plt.plot(hist_2.history['val_accuracy'], label='Modelo 2 (SGD / Batch 128)', marker='s')
plt.title('Comparación de Hiperparámetros (Precisión de Validación)')
plt.xlabel('Época')
plt.ylabel('Precisión')
plt.legend()
plt.grid(True)
plt.show()

### Análisis de los resultados

* **El impacto del Optimizador:** Normalmente verás que `Adam` (Modelo 1) toma la delantera muy rápido en las primeras épocas. Esto pasa porque ajusta su tasa de aprendizaje de forma inteligente, mientras que `SGD` es más tradicional y puede requerir muchas más épocas para alcanzar la misma precisión.
* **El tamaño del Batch:** El Modelo 2 usa un `batch_size` de 128. Esto hace que las épocas se procesen más rápido en tu computadora, pero al hacer menos actualizaciones de pesos por época, el aprendizaje puede ser menos preciso o más inestable. A veces, "más rápido" no significa "mejor".
* **Conclusión:** La gráfica demuestra que no basta con armar bien las capas; la configuración de cómo le enseñamos a la red es vital. En la práctica, esto requiere prueba y error hasta dar con la combinación que logre la mayor precisión en el menor tiempo.

## C. Aplicar técnicas de optimización y regularización

Cuando entrenamos redes neuronales, un problema muy común es el **sobreajuste (overfitting)**: el modelo memoriza los datos de entrenamiento a la perfección, pero se equivoca mucho cuando le mostramos imágenes nuevas. Para evitarlo, aplicamos técnicas de regularización.

En este bloque implementaremos dos de las más utilizadas en la industria:
1. **Dropout:** Durante el entrenamiento, esta técnica "apaga" aleatoriamente un porcentaje de las neuronas en cada iteración. Esto obliga a la red a no depender de unas pocas conexiones específicas y a aprender características más generales y robustas de la ropa.
2. **Early Stopping (Parada Temprana):** Monitorea el error de validación en cada época. Si el modelo deja de mejorar, detiene el entrenamiento automáticamente. Esto evita que la red siga dando vueltas memorizando ruido y nos ahorra tiempo de cómputo.

In [ ]:
# ── C. Dropout + Early Stopping ───────────────────────────────────────────────
# NOTA: Requiere que la celda de carga de datos haya sido ejecutada primero.

from tensorflow.keras import callbacks

# 1. Definimos la arquitectura integrando Dropout
model_opt = models.Sequential([
    layers.Input(shape=(28, 28)),
    layers.Reshape((28, 28, 1)),

    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),  # Apagamos el 25% de las neuronas en esta capa

    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),  # Apagamos otro 25%

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),   # En capas densas el porcentaje suele ser mayor (50%)
    layers.Dense(10, activation='softmax')
])

model_opt.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])

# 2. Configuramos el Early Stopping
# patience=3: Le damos un margen de 3 épocas sin mejora antes de rendirse
# restore_best_weights=True: Nos asegura recuperar la mejor versión del modelo
early_stopping = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True,
    verbose=1
)

# 3. Entrenamos el modelo
print("Iniciando entrenamiento con Dropout y Early Stopping...")
# Ponemos 20 épocas de máximo; el Early Stopping lo cortará antes si es necesario
hist_opt = model_opt.fit(
    train_images,
    train_labels,
    epochs=20,
    batch_size=128,          # Batch más grande para acelerar el entrenamiento
    validation_split=0.2,
    callbacks=[early_stopping],
    verbose=1
)

# 4. Evaluación final
test_loss_opt, test_acc_opt = model_opt.evaluate(test_images, test_labels, verbose=0)
print(f"\nPrecisión final del modelo optimizado en Test: {test_acc_opt:.4f}")

### Análisis de los resultados

* **Impacto del Dropout:** Al agregar cortes aleatorios en las conexiones, el entrenamiento es un poco más lento en alcanzar valores altos de precisión inicial, pero a cambio, la curva de validación se mantiene mucho más estable y cercana a la de entrenamiento.
* **Eficiencia del Early Stopping:** Aunque configuramos el modelo para 20 épocas, es muy probable que el proceso se haya detenido alrededor de la época 8 o 10. Recuperar los mejores pesos (`restore_best_weights=True`) es vital, ya que evita que nos quedemos con la última época procesada, que suele estar más sobreajustada.
* **Generalización:** La precisión final en los datos de prueba será más confiable. En la vida real, preferimos un modelo con un 88% de precisión estable y real, que uno con 99% en entrenamiento pero que colapsa con datos nuevos.